In [1]:
import pickle
import torch
import numpy as np
from statistics import mean
import torch.distributions as dist
import torch.nn as nn
from torch.nn.parameter import Parameter
from torch.nn import init
import functorch
import scanpy as sc
import leidenalg
import umap
import anndata as ad
import scvelo as scv
import seaborn as sns
from matplotlib import pyplot as plt
import pandas as pd
import sys
sys.path.append('/home/ylz0045/Singlecell/RNAVelocity/DeepKINET_2024/src/deepkinet')
import workflow
import utils
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

## Gastrulation erythroid

In [155]:
adata = sc.read_h5ad('../Data/gastrulation_erythroid.h5ad')
raw_adata = sc.read_h5ad('../Data/gastrulation_erythroid.h5ad')
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
adata.layers['spliced'] = raw_adata[:, adata.var_names].layers['spliced']
adata.layers['unspliced'] = raw_adata[:, adata.var_names].layers['unspliced']

adata, deepkinet_exp = workflow.estimate_kinetics(adata)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

scv.tl.velocity_graph(adata, vkey='splicing_rate', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='splicing_rate')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata1, basis='umap', save = False, vkey='splicing_rate', color='celltype', title='DeepKINET', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Gastrulation_Erythroid_Results/DeepKINET_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Gastrulation_Erythroid_Results/DeepKINET.h5ad')

Filtered out 47456 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Extracted 3000 highly variable genes.
Logarithmized X.
make_sample_one_hot_mat
batch_key is None
loss_mode poisson
Start Dynamics opt
Dynamics opt patience 10
val loss mean (post 10 epochs) at epoch 0 is 3.658844232559204
Early Stopping! at 27 epoch
Done Dynamics opt
Dynamics_last_val_loss:3.422959804534912
Dynamics_last_test_loss:6.806799411773682
Start Kinetics opt
Kinetics opt patience 10
val loss mean (post 10 epochs) at epoch 0 is 3.4149224758148193
Early Stopping! at 22 epoch
Done Kinetics opt
Kinetics_last_val_loss:3.4120609760284424
Kinetics_last_test_loss:6.784541130065918
train_s_correlation 0.2539530143856269
train_u_correlation 0.11153191328550897
val_s_correlation 0.24198865423704455
val_u_correlation 0.06779199659503625
test_s_correlation 0.24510689642750186
test_u_correlation 0.06720217574503813
Normalized count data: spliced, unspliced.
computing neighbors
    fi

  0%|          | 0/9815 [00:00<?, ?cells/s]

    finished (0:00:34) --> added 
    'splicing_rate_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:03) --> added
    'splicing_rate_umap', embedded velocity vectors (adata.obsm)


## Dentate Gyrus neurogenesis

In [156]:
adata = sc.read_h5ad('../Data/DentateGyrus_10X43_1.h5ad')
raw_adata = sc.read_h5ad('../Data/DentateGyrus_10X43_1.h5ad')
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3000)
adata.layers['spliced'] = raw_adata[:, adata.var_names].layers['spliced']
adata.layers['unspliced'] = raw_adata[:, adata.var_names].layers['unspliced']

adata, deepkinet_exp = workflow.estimate_kinetics(adata)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

scv.tl.velocity_graph(adata, vkey='splicing_rate', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='splicing_rate')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata1, basis='umap', save = False, vkey='splicing_rate', color='clusters', title='DeepKINET', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Dentate_Gyrus_Results/DeepKINET_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Dentate_Gyrus_Results/DeepKINET.h5ad')

Filtered out 10340 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Extracted 3000 highly variable genes.
Logarithmized X.
make_sample_one_hot_mat
batch_key is None
loss_mode poisson
Start Dynamics opt
Dynamics opt patience 10
val loss mean (post 10 epochs) at epoch 0 is 8.849750518798828
Early Stopping! at 85 epoch
Done Dynamics opt
Dynamics_last_val_loss:7.789328098297119
Dynamics_last_test_loss:15.603482246398926
Start Kinetics opt
Kinetics opt patience 10
val loss mean (post 10 epochs) at epoch 0 is 7.807538032531738
Early Stopping! at 21 epoch
Done Kinetics opt
Kinetics_last_val_loss:7.787694454193115
Kinetics_last_test_loss:15.589895248413086
train_s_correlation 0.2574166685491258
train_u_correlation 0.18042484742954712
val_s_correlation 0.19222829713510176
val_u_correlation 0.082865900989027
test_s_correlation 0.2025027664704765
test_u_correlation 0.08877264397021363
Normalized count data: spliced, unspliced.
computing neighbors
    finis

  0%|          | 0/2930 [00:00<?, ?cells/s]

    finished (0:00:20) --> added 
    'splicing_rate_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:01) --> added
    'splicing_rate_umap', embedded velocity vectors (adata.obsm)


## Simulated data

In [158]:
adata = sc.read_h5ad('../Data/simulated_T_cell_expint.h5ad')
raw_adata = sc.read_h5ad('../Data/simulated_T_cell_expint.h5ad')
adata.obsm['X_umap'] = pickle.load(open('../Data/UMAP.pkl','rb'))
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=3)
adata.layers['spliced'] = raw_adata[:, adata.var_names].layers['spliced']
adata.layers['unspliced'] = raw_adata[:, adata.var_names].layers['unspliced']

adata, deepkinet_exp = workflow.estimate_kinetics(adata)
scv.pp.moments(adata, n_neighbors=30)

scv.tl.velocity_graph(adata, vkey='splicing_rate', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='splicing_rate')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='splicing_rate', color='time', title='DeepKINET', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Simulation_Results/DeepKINET_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Simulation_Results/DeepKINET.h5ad')

Normalized count data: X, spliced, unspliced.
Extracted 3 highly variable genes.
make_sample_one_hot_mat
batch_key is None
loss_mode poisson
Start Dynamics opt
Dynamics opt patience 10
val loss mean (post 10 epochs) at epoch 0 is 0.058433640748262405
Early Stopping! at 80 epoch
Done Dynamics opt
Dynamics_last_val_loss:0.031010296195745468
Dynamics_last_test_loss:0.061342209577560425
Start Kinetics opt
Kinetics opt patience 10
val loss mean (post 10 epochs) at epoch 0 is 0.03146436810493469
Early Stopping! at 28 epoch
Done Kinetics opt
Kinetics_last_val_loss:0.031082218512892723
Kinetics_last_test_loss:0.0610014945268631
train_s_correlation 0.8796612750731364
train_u_correlation 0.8350561044507022
val_s_correlation 0.867427909971248
val_u_correlation 0.8184581621911816
test_s_correlation 0.9122915766379318
test_u_correlation 0.805677468616866
Normalized count data: spliced, unspliced.
computing neighbors
    finished (0:00:01) --> added 
    'distances' and 'connectivities', weighted ad

  0%|          | 0/4000 [00:00<?, ?cells/s]

    finished (0:00:02) --> added 
    'splicing_rate_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:01) --> added
    'splicing_rate_umap', embedded velocity vectors (adata.obsm)
